# Notebook 2 — Coder Monte Carlo de zéro

**Chapitre 17 — Monte Carlo, TD-Learning, TD(λ), SARSA et Q-Learning**

**Vous allez implémenter pas à pas :**
1. La méthode **First-Visit Monte Carlo** pour évaluer une politique aléatoire sur un GridWorld.
2. La méthode **Every-Visit Monte Carlo**.
3. Une **comparaison** entre les deux sur un même environnement.
4. Le **contrôle Monte Carlo on-policy** avec exploration epsilon-greedy.

Rappel (Éq. 1) : `V(S_t) <- V(S_t) + alpha [G_t - V(S_t)]`, avec `G_t` le retour total observé jusqu'à la fin de l'épisode.

> Ce notebook est **autonome** : il n'a besoin que de `numpy` (et `matplotlib` pour les figures), déjà installés sur Colab.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

rng = np.random.default_rng(42)
print("Imports OK")

## 0. Environnement GridWorld

Grille `n x n`. L'agent part en haut à gauche `(0,0)`, l'objectif est en bas à droite.
Chaque pas coûte `-1`, atteindre l'objectif donne `0` et termine l'épisode.
Actions : 0=haut, 1=bas, 2=gauche, 3=droite.

In [ ]:
class GridWorld:
    def __init__(self, n=4):
        self.n = n
        self.goal = (n - 1, n - 1)
        self.actions = [0, 1, 2, 3]  # haut, bas, gauche, droite
        self.reset()

    def reset(self):
        self.state = (0, 0)
        return self.state

    def step(self, action):
        r, c = self.state
        if action == 0:
            r = max(0, r - 1)
        elif action == 1:
            r = min(self.n - 1, r + 1)
        elif action == 2:
            c = max(0, c - 1)
        elif action == 3:
            c = min(self.n - 1, c + 1)
        self.state = (r, c)
        done = self.state == self.goal
        reward = 0.0 if done else -1.0
        return self.state, reward, done

    def tous_les_etats(self):
        return [(r, c) for r in range(self.n) for c in range(self.n)]

env = GridWorld(n=4)
print("Etat initial :", env.reset(), "| Objectif :", env.goal)

In [ ]:
def politique_aleatoire(state, env):
    """Choisit une action uniformément au hasard."""
    return int(rng.choice(env.actions))

def generer_episode(env, politique, max_steps=200):
    """Retourne une liste de (state, action, reward)."""
    episode = []
    state = env.reset()
    for _ in range(max_steps):
        action = politique(state, env)
        next_state, reward, done = env.step(action)
        episode.append((state, action, reward))
        state = next_state
        if done:
            break
    return episode

ep = generer_episode(env, politique_aleatoire)
print(f"Exemple d'épisode : {len(ep)} pas")

## 1. First-Visit Monte Carlo

On ne compte le retour `G` que pour la **première** occurrence de chaque état dans l'épisode.

In [ ]:
def mc_first_visit(env, politique, n_episodes=5000, gamma=0.9):
    returns_sum = defaultdict(float)
    returns_count = defaultdict(int)
    V = defaultdict(float)
    for _ in range(n_episodes):
        episode = generer_episode(env, politique)
        etats_vus = set()
        G = 0.0
        # parcours à rebours pour calculer G_t
        for t in reversed(range(len(episode))):
            state, _, reward = episode[t]
            G = gamma * G + reward
            if state not in etats_vus:  # première visite
                etats_vus.add(state)
                returns_sum[state] += G
                returns_count[state] += 1
                V[state] = returns_sum[state] / returns_count[state]
    return V

V_first = mc_first_visit(env, politique_aleatoire, n_episodes=5000, gamma=0.9)
print("V(0,0) =", round(V_first[(0, 0)], 2))
print("V(goal) =", round(V_first[env.goal], 2))

## 2. Every-Visit Monte Carlo

On compte le retour `G` à **chaque** occurrence de l'état.

In [ ]:
def mc_every_visit(env, politique, n_episodes=5000, gamma=0.9):
    returns_sum = defaultdict(float)
    returns_count = defaultdict(int)
    V = defaultdict(float)
    for _ in range(n_episodes):
        episode = generer_episode(env, politique)
        G = 0.0
        for t in reversed(range(len(episode))):
            state, _, reward = episode[t]
            G = gamma * G + reward
            returns_sum[state] += G       # chaque visite compte
            returns_count[state] += 1
            V[state] = returns_sum[state] / returns_count[state]
    return V

V_every = mc_every_visit(env, politique_aleatoire, n_episodes=5000, gamma=0.9)
print("V(0,0) =", round(V_every[(0, 0)], 2))

## 3. Comparaison First-Visit vs Every-Visit

In [ ]:
def grille_valeurs(V, env):
    grid = np.zeros((env.n, env.n))
    for (r, c) in env.tous_les_etats():
        grid[r, c] = V[(r, c)]
    return grid

g_first = grille_valeurs(V_first, env)
g_every = grille_valeurs(V_every, env)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, g, titre in zip(axes, [g_first, g_every], ["First-Visit MC", "Every-Visit MC"]):
    im = ax.imshow(g, cmap="viridis")
    for r in range(env.n):
        for c in range(env.n):
            ax.text(c, r, f"{g[r, c]:.1f}", ha="center", va="center", color="white")
    ax.set_title(titre)
    fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

ecart_max = np.max(np.abs(g_first - g_every))
print(f"Écart maximal entre les deux estimations : {ecart_max:.3f}")
print("-> Les deux convergent vers des valeurs proches quand n_episodes augmente.")

## 4. Contrôle Monte Carlo on-policy (epsilon-greedy)

On apprend maintenant `Q(s,a)` et on **améliore la politique** : exploration aléatoire avec probabilité `epsilon`, sinon action gloutonne.

In [ ]:
def politique_epsilon_greedy(state, Q, env, epsilon):
    if rng.random() < epsilon:
        return int(rng.choice(env.actions))
    q_vals = [Q[(state, a)] for a in env.actions]
    return int(np.argmax(q_vals))

def mc_controle_on_policy(env, n_episodes=20000, gamma=0.9, epsilon=0.1):
    Q = defaultdict(float)
    returns_sum = defaultdict(float)
    returns_count = defaultdict(int)
    for _ in range(n_episodes):
        # générer un épisode avec la politique epsilon-greedy courante
        episode = []
        state = env.reset()
        for _ in range(200):
            action = politique_epsilon_greedy(state, Q, env, epsilon)
            next_state, reward, done = env.step(action)
            episode.append((state, action, reward))
            state = next_state
            if done:
                break
        # mise à jour first-visit sur les paires (s, a)
        vus = set()
        G = 0.0
        for t in reversed(range(len(episode))):
            s, a, r = episode[t]
            G = gamma * G + r
            if (s, a) not in vus:
                vus.add((s, a))
                returns_sum[(s, a)] += G
                returns_count[(s, a)] += 1
                Q[(s, a)] = returns_sum[(s, a)] / returns_count[(s, a)]
    return Q

Q = mc_controle_on_policy(env, n_episodes=20000, gamma=0.9, epsilon=0.1)
print("Apprentissage terminé.")

In [ ]:
fleches = {0: "^", 1: "v", 2: "<", 3: ">"}
print("Politique optimale apprise (action gloutonne par état) :\n")
for r in range(env.n):
    ligne = ""
    for c in range(env.n):
        if (r, c) == env.goal:
            ligne += " G "
        else:
            best = int(np.argmax([Q[((r, c), a)] for a in env.actions]))
            ligne += f" {fleches[best]} "
    print(ligne)
print("\n-> L'agent doit globalement aller vers la droite et vers le bas (objectif en bas-droite).")

## 5. À vous de jouer

1. Changez `gamma` (0.5, 0.9, 0.99) et observez l'impact sur `V`.
2. Changez `epsilon` (0.01, 0.1, 0.3) dans le contrôle : trop petit = peu d'exploration, trop grand = politique bruitée.
3. Augmentez `n=6` dans `GridWorld` : combien d'épisodes faut-il pour une bonne politique ?
4. **Question finale :** sur quel type d'environnement Monte Carlo est-il adapté ? Justifiez avec les études de cas du chapitre (épisodes complets et indépendants).